# SFT 概念与 Wordle 任务

欢迎来到 Qwen3-1.7B 训练实战教程的第一阶段——监督微调（Supervised Fine-Tuning，SFT）。

本教程的目标是：**训练一个会玩 Wordle 猜词游戏的语言模型**。这是一个两阶段的任务——首先通过 SFT 让模型学会输出符合格式要求的 Wordle 猜测，再通过 RL（强化学习）让模型学会策略性地利用游戏反馈提高猜中率。

本章是整个两阶段训练的第一站：打好 SFT 的概念基础，理解 Wordle 任务。

---

## 前置要求

为了充分掌握本章内容，你应已具备以下能力：

- 了解 PyTorch 模型训练的基本流程（forward → loss → backward → update）。
- 了解 Transformer 模型的基本结构（attention、FFN、layer norm）。

无须具备 Wordle、SFT 或分布式训练框架经验。

---

## 章节目标

完成本章后，你将能够：

- 理解 Wordle 猜词游戏的规则和 SFT + RL 两阶段训练方案的设计动机。
- 理解 SFT 的核心概念：对话数据格式、Cross-Entropy Loss、Label Masking。
- 理解本课程为何先用 SFT 学习示例轨迹，再用 RL 优化交互策略。

---

## 本章内容

- [1.1 章节介绍](01.01_chapter_intro.ipynb)
- [1.2 Wordle 任务介绍](01.02_wordle_task_intro.ipynb)
- [1.3 SFT 概念与原理](01.03_sft_concepts.ipynb)
- [1.4 章节练习](01.04_chapter_practice.ipynb)

---

## 为什么需要两阶段训练？

训练模型玩 Wordle 是一个多轮决策任务：模型根据游戏反馈（哪些字母对、哪些字母位置不对）逐步缩小候选词范围，最终猜出目标单词。


```text
Wordle 游戏示例：
  目标词：APPLE
  第1轮：模型猜 CRANE → 反馈 X-X-Y-X-G → 调整策略
  第2轮：模型猜 PLANE → 反馈 Y-Y-Y-X-G → 再调整
  ...
  第N轮：模型猜 APPLE → 猜对！
```


Wordle 的下一轮猜测取决于此前反馈，不能只看一个固定答案。只要数据包含高质量的“历史状态 → 下一步猜测”轨迹，SFT 就能进行行为克隆。本课程先用 SFT 学习这些轨迹和输出格式，再用 RL 优化策略。

1. **SFT 阶段（本教程）**：用已有的 Wordle 游戏记录（对话数据）训练模型，让它学会：
   - 模仿数据中的完整回复格式：`<think>...</think><guess>[word]</guess>`。
   - 学习训练轨迹中反馈与下一次猜测之间的关联。
   - 获得由训练标签质量约束、但可能产生一定泛化的初始行为。

2. **RL 阶段（下一教程）**：让模型在实际游戏中试错，通过奖励函数优化多轮决策策略，以提高猜中率为目标。

> **SFT 的角色定位**：在本课程使用的数据、模型和评测设置下，SFT 先降低格式错误并模仿已有轨迹，RL 再根据环境奖励优化策略。SFT 是否改善规则遵守和猜中率，必须分别通过后续评测验证，不能只由训练 loss 推断。